# Environment / Dependencies Setup
One command installs everything the lab needs:

In [ ]:
!pip install -q "pymongo[srv,tls]==4.10.1" python-dotenv==1.0.1 certifi

### Step 1 — Connect to MongoDB
The connection pattern is simple — `load_dotenv()` reads the connection string from `.env` and `certifi.where()` supplies trusted CA certificates.

In [ ]:
import os
import certifi
from dotenv import load_dotenv
import pymongo

# Load the Atlas connection string from the .env file
load_dotenv()
uri = os.environ["MONGODB_URI"]

# Connect to the real MongoDB Atlas cluster
client = pymongo.MongoClient(uri, tlsCAFile=certifi.where())

# Access the database and collections
db = client["school_db"]
courses = db["courses"]
enrollments = db["enrollments"]

print("Connected to MongoDB Atlas")

### Step 2 — Populate Courses and Enrollments
Inserts courses with `seats_total` / `seats_available` and a set of enrollment documents.

In [ ]:
from datetime import datetime

courses.drop()
enrollments.drop()

# Insert 4 courses with deliberately small seat counts so we can
# realistically demonstrate a course filling up and a transaction aborting
course_records = [
    {"course_id": "CS101",   "title": "Introduction to Computer Science",
     "seats_total": 3, "seats_available": 3},
    {"course_id": "CS201",   "title": "Data Structures and Algorithms",
     "seats_total": 5, "seats_available": 5},
    {"course_id": "MATH101", "title": "Calculus I",
     "seats_total": 2, "seats_available": 2},
    {"course_id": "PHYS101", "title": "General Physics I",
     "seats_total": 4, "seats_available": 4},
]
courses.insert_many(course_records)
print(f"Inserted {len(course_records)} courses.")

# Enrollments starts empty — we will add enrollment documents in later steps
print(f"Enrollments: {enrollments.count_documents({})} documents.")

### Step 3 — The Problem: Non-Atomic Enrollment
Decrements a seat then inserts an enrollment as two separate writes — they can diverge if one fails.

In [ ]:
# Two separate operations: decrement seats, then insert the enrollment record
cid = "CS101"

courses.update_one(
    {"course_id": cid, "seats_available": {"$gt": 0}},
    {"$inc": {"seats_available": -1}}
)
print(f"Step 3a: decremented seats_available for {cid}")

enrollments.insert_one({
    "enrollment_id": "ENR999",
    "student_id": "STU999",
    "student_name": "Test Student",
    "course_id": cid,
    "enrolled_at": datetime.utcnow()
})
print(f"Step 3b: inserted enrollment document for STU999")

# Show the current state
course_after = courses.find_one({"course_id": cid}, {"_id": 0})
enr_count = enrollments.count_documents({"course_id": cid})
print(f"\nCurrent state of {cid}: seats_available = {course_after['seats_available']}, enrollments = {enr_count}")

### Step 4 — The Fix: Multi-Document Transaction
Wraps the seat decrement and enrollment insert in one session and transaction, so both commit or neither does.

In [ ]:
# Reset CS101 back to full capacity for a clean demo
courses.update_one(
    {"course_id": "CS101"},
    {"$set": {"seats_available": 3}}
)
enrollments.delete_many({"course_id": "CS101"})

def enroll_student(session, cid, student_id, student_name):
    """Atomically decrement a seat and insert the enrollment record."""
    result = courses.find_one_and_update(
        {"course_id": cid, "seats_available": {"$gt": 0}},
        {"$inc": {"seats_available": -1}},
        session=session
    )
    if result is None:
        raise Exception(f"No seats available in {cid}")
    enrollments.insert_one({
        "enrollment_id": f"ENR{student_id[-3:]}",
        "student_id": student_id,
        "student_name": student_name,
        "course_id": cid,
        "enrolled_at": datetime.utcnow()
    }, session=session)

# Enroll 3 students into CS101 (which has 3 seats) using transactions
students_to_enroll = [
    ("STU001", "Alice Johnson"),
    ("STU002", "Bob Smith"),
    ("STU003", "Charlie Brown"),
]

for sid, sname in students_to_enroll:
    with client.start_session() as session:
        session.with_transaction(
            lambda s: enroll_student(s, "CS101", sid, sname)
        )
    print(f"Enrolled {sname} ({sid}) in CS101")

# Verify the state
cs101 = courses.find_one({"course_id": "CS101"}, {"_id": 0})
enr = list(enrollments.find({"course_id": "CS101"}, {"_id": 0, "enrollment_id": 0}))
print(f"\nCS101 seats_available: {cs101['seats_available']}")
print(f"CS101 enrollments ({len(enr)}):")
for e in enr:
    print(f"  {e['student_id']}: {e['student_name']}")

### Step 5 — Abort Path: Enrolling Into a Full Course
Tries to enroll in a full course; the transaction aborts and the database is left unchanged.

In [ ]:
# CS101 now has 0 seats_available — try to enroll one more student
try:
    with client.start_session() as session:
        session.with_transaction(
            lambda s: enroll_student(s, "CS101", "STU999", "Overflow Student")
        )
    print("ERROR: This line should not print — enrollment should have failed.")
except Exception as e:
    print(f"Transaction aborted as expected: {e}")

# Verify nothing changed
cs101 = courses.find_one({"course_id": "CS101"}, {"_id": 0})
overflow = enrollments.find_one({"student_id": "STU999"})
print(f"\nAfter attempted overflow:")
print(f"  seats_available: {cs101['seats_available']}")
print(f"  STU999 enrollment exists: {overflow is not None}")

### Step 6 — Change Streams: Watching Enrollments in Real Time
Opens a change stream on `enrollments` to observe committed writes as they happen.

In [ ]:
# Open a change stream on the enrollments collection
pipeline = []
change_stream = enrollments.watch(pipeline)

# Perform one more enrollment to generate a change event
with client.start_session() as session:
    session.with_transaction(
        lambda s: enroll_student(s, "CS201", "STU010", "Diana Prince")
    )

# Read the change event from the stream
event = change_stream.next()

print(f"Change stream event:")
print(f"  operationType: {event['operationType']}")
print(f"  fullDocument.student_id: {event['fullDocument']['student_id']}")
print(f"  fullDocument.student_name: {event['fullDocument']['student_name']}")
print(f"  fullDocument.course_id: {event['fullDocument']['course_id']}")

change_stream.close()

### Step 7 — Lightweight Backup and Restore from Python
Exports both collections to a JSON file as a lightweight backup.

In [ ]:
import json

# --- Backup: export every document in both collections to a JSON file ---
backup = {
    "courses": [doc for doc in courses.find({}, {"_id": 0})],
    "enrollments": [doc for doc in enrollments.find({}, {"_id": 0})],
}

backup_path = "school_db_backup.json"
with open(backup_path, "w") as f:
    json.dump(backup, f, indent=2, default=str)

print(f"Backup saved to {backup_path}")
print(f"  courses: {len(backup['courses'])} documents")
print(f"  enrollments: {len(backup['enrollments'])} documents")

Wipes both collections and reinserts from the JSON backup to restore the state.

In [ ]:
# --- Restore: wipe both collections and reinsert from the JSON backup ---
courses.drop()
enrollments.drop()
print(f"Dropped collections. courses: {courses.count_documents({})}, enrollments: {enrollments.count_documents({})}")

with open(backup_path, "r") as f:
    restored = json.load(f)

if restored["courses"]:
    courses.insert_many(restored["courses"])
if restored["enrollments"]:
    enrollments.insert_many(restored["enrollments"])

print(f"Restored. courses: {courses.count_documents({})}, enrollments: {enrollments.count_documents({})}")

### Step 8 — Summary Report
Collects course counts, remaining seats, and total enrollments into one report.

In [ ]:
total_courses = courses.count_documents({})
total_enrollments = enrollments.count_documents({})

print("       ENROLLMENT TRANSACTIONS")
print(f"\nTotal courses: {total_courses}")
print(f"Total enrollments: {total_enrollments}")

print("\nSeats remaining per course:")
for c in courses.find({}, {"_id": 0}).sort("course_id", 1):
    enr_count = enrollments.count_documents({"course_id": c["course_id"]})
    print(f"  {c['course_id']}: {c['seats_available']}/{c['seats_total']} seats available ({enr_count} enrolled)")